# Why infrastructure as code

What infrastructure as code is and the problems it solves, how Terraform records what it manages, and how much of Tallybook's cloud was created by hand and forgotten.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/terraform-infrastructure-as-code/why-infrastructure-as-code). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

In the cloud course, Tallybook found idle servers, disks attached to nothing and firewall rules nobody could explain. In the Linux course, one of those rules (SSH open to the whole internet, "temporary, for the 2025 migration") let attackers in. Every one of these was created by someone clicking in the cloud console. There was no record of who made it, why, or whether it was still needed.

Part of Tallybook's infrastructure is managed differently: written as code with **Terraform**, reviewed in pull requests, and applied by a pipeline. This course teaches you to work that way, and to review infrastructure changes safely, using Tallybook's real Terraform files.

## The concept

### ClickOps and its problems

Changing infrastructure by hand in a console ("ClickOps") is quick once, and costly forever:

- no record of who changed what, or why;
- no review before a change;
- environments drift apart (staging stops matching production);
- rebuilding after a disaster depends on memory.

### Infrastructure as code (IaC)

Infrastructure is described in text files, kept in git, changed through reviewed pull requests, and applied by a tool. **Terraform** is the most widely used. You write what you want, for example "four API servers of this size", and Terraform works out what to create, change or delete to get there.

![Top: ClickOps, an engineer clicking changes straight into the cloud, with no record, review or reproducibility. Bottom: infrastructure as code, a change edited in a .tf file, reviewed in a pull request with its plan, applied by Terraform and recorded in git and state.](https://academy.cloudtechanalytics.com/images/courses/terraform/clickops-vs-iac.svg "ClickOps versus infrastructure as code.")

### State

Terraform keeps a **state file** recording every resource it manages and its last known settings. Anything that isn't in the state is invisible to Terraform: it won't change it, review it, or delete it.

## Example

Tallybook's state file is JSON. Load it, and list what Terraform manages:

In [ ]:
import json
from urllib.request import urlopen

import pandas as pd

def load(url):
    with urlopen(url) as f:
        return json.load(f)

state = load("https://academy.cloudtechanalytics.com/datasets/terraform/terraform.tfstate")
managed = pd.DataFrame([
    {"address": f"{r['type']}.{r['name']}", "type": r["type"], "id": inst["attributes"]["id"]}
    for r in state["resources"] for inst in r["instances"]
])
print("Terraform manages", len(managed), "resources")
managed["type"].value_counts()

*Expected output:*

```
Terraform manages 37 resources
type
aws_instance               19
aws_security_group_rule    10
aws_s3_bucket               4
aws_lb                      2
aws_db_instance             2
Name: count, dtype: int64
```

Now compare with everything that's really in the cloud account, from the cloud course's inventory. Resource IDs link the two:

In [ ]:
inventory = pd.read_csv("https://academy.cloudtechanalytics.com/datasets/cloud/resources.csv")
main_types = inventory[inventory["type"].isin(["vm", "database", "load_balancer", "bucket"])].copy()
main_types["in_terraform"] = main_types["resource_id"].isin(managed["id"])
print(pd.crosstab(main_types["type"], main_types["in_terraform"]))

unmanaged = main_types[~main_types["in_terraform"]]
print("Monthly cost of running servers Terraform doesn't know about: $",
      round(unmanaged.loc[unmanaged["status"] == "running", "hourly_usd"].sum() * 730, 2))

*Expected output:*

```
in_terraform   False  True
type
bucket             2      4
database           0      2
load_balancer      0      2
vm                19     19
Monthly cost of running servers Terraform doesn't know about: $ 876.0
```

Every database and load balancer is in Terraform. But half the servers aren't, and among them are the development machines, the idle "test" servers and the stopped ones from the cloud course. The public bucket of customer files isn't either. The resources nobody manages as code are exactly the ones that were forgotten.

## Walkthrough

1. Run the cells. List the names of the unmanaged servers and buckets. Which teams made them?
2. Find the ID of `prod-db` in the inventory and check it's in the state.
3. Open the state file in a text editor (download it from the URL). What else is in it besides resources?
4. Write down three changes from the cloud and Linux courses that would have been caught by a pull request review.

## Practice

**Practice:** How many resources does Terraform manage (resource instances in the state)?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** How many **servers (VMs)** in the cloud account are **not** in Terraform?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding